In [ ]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import OneHotEncoder
#LSTM
# Load and preprocess the dataset
data = pd.read_csv('embedded_sbert.csv')
data['embedding'] = data['embedding'].apply(lambda x: np.fromstring(x[1:-1], sep=','))
X = np.array(data['embedding'].tolist())

# Define target columns and class counts
target_columns = {
    'provokingviolence': 4,
    'individualharrassment': 4,
    'emotionaldistress': 3
}

# One-hot encode each target column
encoded_targets = {}
for col, num_classes in target_columns.items():
    encoder = OneHotEncoder(sparse_output=False)
    encoded_targets[col] = encoder.fit_transform(data[col].values.reshape(-1, 1))

# Concatenate one-hot encoded targets into a single array
y = np.hstack([encoded_targets[col] for col in target_columns])

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = X_train[:, np.newaxis, :]
X_val = X_val[:, np.newaxis, :]

# Custom Dataset class
class MultiOutputDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input': torch.tensor(self.features[idx], dtype=torch.float32),
            'label': torch.tensor(self.labels[idx], dtype=torch.float32)
        }

# DataLoaders
train_dataset = MultiOutputDataset(X_train, y_train)
val_dataset = MultiOutputDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Modified LSTM Model with Additional Layers and Batch Normalization
class MultiOutputLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dims, num_layers=2):
        super(MultiOutputLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.3)

        # Batch Normalization
        self.batch_norm = nn.BatchNorm1d(hidden_dim)

        # Separate output layers for each target
        self.output_heads = nn.ModuleDict({
            target: nn.Linear(hidden_dim, output_dim) for target, output_dim in output_dims.items()
        })

        # Dropout
        self.dropout = nn.Dropout(0.4)

    def forward(self, x):
        _, (hn, _) = self.lstm(x)
        x = self.batch_norm(hn[-1])  # Batch normalization
        x = self.dropout(x)  # Dropout

        outputs = {target: head(x) for target, head in self.output_heads.items()}
        return outputs

# Model parameters
input_dim = X.shape[1]
hidden_dim = 128
output_dims = {target: num_classes for target, num_classes in target_columns.items()}
model = MultiOutputLSTM(input_dim, hidden_dim, output_dims)

# Loss function, optimizer, and learning rate scheduler
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.7)

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Training function with scheduler step
def train_model(model, train_loader, criterion, optimizer, scheduler, epochs=10):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            input_data = batch['input'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_data)
            losses = [criterion(outputs[target], labels[:, start:end])
                      for target, (start, end) in zip(target_columns.keys(),
                                                      [(0,4), (4,8), (8,11)])]
            loss = sum(losses)
            total_loss += loss.item()

            loss.backward()
            optimizer.step()

        scheduler.step()
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {total_loss / len(train_loader):.4f}")

# Train the model
train_model(model, train_loader, criterion, optimizer, scheduler, epochs=10)

# Evaluation function
def evaluate_model(model, val_loader):
    model.eval()
    predictions, true_labels = {}, {}

    for target in target_columns.keys():
        predictions[target], true_labels[target] = [], []

    with torch.no_grad():
        for batch in val_loader:
            input_data = batch['input'].to(device)
            labels = batch['label'].cpu().numpy()

            outputs = model(input_data)
            for target, pred in outputs.items():
                predictions[target].append(pred.cpu().numpy())
                start, end = (0,4) if target == 'provokingviolence' else ((4,8) if target == 'individualharrassment' else (8,11))
                true_labels[target].append(labels[:, start:end])

    predictions = {target: np.vstack(preds) for target, preds in predictions.items()}
    true_labels = {target: np.vstack(labels) for target, labels in true_labels.items()}

    return predictions, true_labels

# Evaluate the model
predictions, true_labels = evaluate_model(model, val_loader)

# Apply sigmoid and calculate metrics for each output head
for target, num_classes in target_columns.items():
    y_pred_binary = (torch.sigmoid(torch.tensor(predictions[target])) > 0.5).int().numpy()
    y_true_binary = true_labels[target]

    y_pred_labels = np.argmax(y_pred_binary, axis=1)
    y_true_labels = np.argmax(y_true_binary, axis=1)

    print(f"Classification report for {target}:")
    print(classification_report(y_true_labels, y_pred_labels))

    overall_accuracy = accuracy_score(y_true_labels, y_pred_labels)
    print(f"Overall Accuracy for {target}: {overall_accuracy:.4f}\n")


Epoch 1/10, Loss: 1.2678
Epoch 2/10, Loss: 1.1750
Epoch 3/10, Loss: 1.1629
Epoch 4/10, Loss: 1.1546
Epoch 5/10, Loss: 1.1499
Epoch 6/10, Loss: 1.1470
Epoch 7/10, Loss: 1.1407
Epoch 8/10, Loss: 1.1381
Epoch 9/10, Loss: 1.1358
Epoch 10/10, Loss: 1.1300
Classification report for provokingviolence:
              precision    recall  f1-score   support

           0       0.40      0.46      0.42      1975
           1       0.00      0.00      0.00       966
           2       0.66      0.76      0.70      5855
           3       0.77      0.68      0.72      2191

    accuracy                           0.62     10987
   macro avg       0.46      0.47      0.46     10987
weighted avg       0.57      0.62      0.60     10987

Overall Accuracy for provokingviolence: 0.6227

Classification report for individualharrassment:
              precision    recall  f1-score   support

           0       0.01      0.19      0.01        81
           1       0.58      0.29      0.38      2386
         

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, accuracy_score

# Function to calculate and display all metrics for each target
def display_metrics(y_true_labels, y_pred_labels, target):
    print(f"Classification report for {target}:")
    # Classification report (includes precision, recall, f1-score for each class)
    print(classification_report(y_true_labels, y_pred_labels, zero_division=0))

    # Overall Accuracy
    overall_accuracy = accuracy_score(y_true_labels, y_pred_labels)
    print(f"Overall Accuracy for {target}: {overall_accuracy:.4f}")

    # Micro, Macro, and Weighted Metrics
    print(f"Micro Precision for {target}: {precision_score(y_true_labels, y_pred_labels, average='micro'):.4f}")
    print(f"Macro Precision for {target}: {precision_score(y_true_labels, y_pred_labels, average='macro'):.4f}")
    print(f"Weighted Precision for {target}: {precision_score(y_true_labels, y_pred_labels, average='weighted'):.4f}")

    print(f"Micro Recall for {target}: {recall_score(y_true_labels, y_pred_labels, average='micro'):.4f}")
    print(f"Macro Recall for {target}: {recall_score(y_true_labels, y_pred_labels, average='macro'):.4f}")
    print(f"Weighted Recall for {target}: {recall_score(y_true_labels, y_pred_labels, average='weighted'):.4f}")

    print(f"Micro F1-score for {target}: {f1_score(y_true_labels, y_pred_labels, average='micro'):.4f}")
    print(f"Macro F1-score for {target}: {f1_score(y_true_labels, y_pred_labels, average='macro'):.4f}")
    print(f"Weighted F1-score for {target}: {f1_score(y_true_labels, y_pred_labels, average='weighted'):.4f}\n")

# Evaluate the model
predictions, true_labels = evaluate_model(model, val_loader)

# Apply sigmoid and calculate metrics for each output head
for target, num_classes in target_columns.items():
    y_pred_binary = (torch.sigmoid(torch.tensor(predictions[target])) > 0.5).int().numpy()
    y_true_binary = true_labels[target]

    # Convert one-hot encoded predictions and true labels back to single class labels
    y_pred_labels = np.argmax(y_pred_binary, axis=1)
    y_true_labels = np.argmax(y_true_binary, axis=1)

    # Display metrics for the current target
    display_metrics(y_true_labels, y_pred_labels, target)


Classification report for provokingviolence:
              precision    recall  f1-score   support

           0       0.40      0.46      0.42      1975
           1       0.00      0.00      0.00       966
           2       0.66      0.76      0.70      5855
           3       0.77      0.68      0.72      2191

    accuracy                           0.62     10987
   macro avg       0.46      0.47      0.46     10987
weighted avg       0.57      0.62      0.60     10987

Overall Accuracy for provokingviolence: 0.6227
Micro Precision for provokingviolence: 0.6227
Macro Precision for provokingviolence: 0.4550
Weighted Precision for provokingviolence: 0.5746
Micro Recall for provokingviolence: 0.6227
Macro Recall for provokingviolence: 0.4749
Weighted Recall for provokingviolence: 0.6227
Micro F1-score for provokingviolence: 0.6227
Macro F1-score for provokingviolence: 0.4629
Weighted F1-score for provokingviolence: 0.5959

Classification report for individualharrassment:
            

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Micro Precision for individualharrassment: 0.4310
Macro Precision for individualharrassment: 0.4379
Weighted Precision for individualharrassment: 0.5706
Micro Recall for individualharrassment: 0.4310
Macro Recall for individualharrassment: 0.3242
Weighted Recall for individualharrassment: 0.4310
Micro F1-score for individualharrassment: 0.4310
Macro F1-score for individualharrassment: 0.3180
Weighted F1-score for individualharrassment: 0.4553

Classification report for emotionaldistress:
              precision    recall  f1-score   support

           0       0.02      0.02      0.02       100
           1       0.62      0.43      0.51      3151
           2       0.80      0.89      0.84      7736

    accuracy                           0.75     10987
   macro avg       0.48      0.45      0.46     10987
weighted avg       0.74      0.75      0.74     10987

Overall Accuracy for emotionaldistress: 0.7544
Micro Precision for emotionaldistress: 0.7544
Macro Precision for emotionaldist

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support
from sklearn.preprocessing import OneHotEncoder

# Load the dataset
data = pd.read_csv('embedded_sbert.csv')

# Convert the 'embedded_text' column to numpy arrays
data['embedding'] = data['embedding'].apply(lambda x: np.fromstring(x[1:-1], sep=','))

X = np.array(data['embedding'].tolist())

# Define target columns and their respective number of classes
target_columns = {
    'provokingviolence': 4,
    'individualharrassment': 4,
    'emotionaldistress': 3
}

# One-hot encode each target column
encoded_targets = {}
encoders = {}
for col, num_classes in target_columns.items():
    encoder = OneHotEncoder(sparse_output=False, categories='auto')
    encoded = encoder.fit_transform(data[col].values.reshape(-1, 1))
    encoded_targets[col] = encoded
    encoders[col] = encoder  # Save encoder for inverse transformations if needed

# Concatenate one-hot encoded targets into a single array
y = np.hstack([encoded_targets[col] for col in target_columns])

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Reshape X to add a time dimension for hierarchical LSTM (word-level)
X_train = X_train[:, np.newaxis, :]
X_val = X_val[:, np.newaxis, :]

# Create a custom Dataset class
class MultiOutputDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input': torch.tensor(self.features[idx], dtype=torch.float32),
            'label': torch.tensor(self.labels[idx], dtype=torch.float32)
        }

# Create DataLoaders
train_dataset = MultiOutputDataset(X_train, y_train)
val_dataset = MultiOutputDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Define the Hierarchical BiLSTM model with multiple output heads
class HierarchicalBiLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dims, word_num_layers=1, sent_num_layers=1):
        super(HierarchicalBiLSTM, self).__init__()

        self.word_bilstm = nn.LSTM(
            input_dim, hidden_dim, word_num_layers,
            batch_first=True, bidirectional=True
        )

        # Sentence-level BiLSTM to capture sentence-level context
        self.sent_bilstm = nn.LSTM(
            hidden_dim * 2, hidden_dim, sent_num_layers,
            batch_first=True, bidirectional=True
        )

        # Define output heads for each target
        self.output_heads = nn.ModuleDict({
            target: nn.Linear(hidden_dim * 2, output_dim)
            for target, output_dim in output_dims.items()
        })

        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        # x: (batch_size, num_words, word_embedding_dim)
        batch_size, num_words, _ = x.size()

        # Word-level BiLSTM
        word_level_outputs, _ = self.word_bilstm(x)  # (batch_size, num_words, hidden_dim * 2)

        # Get the sentence representation by aggregating word-level outputs
        sentence_embedding = word_level_outputs.mean(dim=1)  # (batch_size, hidden_dim * 2)

        # Sentence-level BiLSTM
        sentence_embedding = sentence_embedding.unsqueeze(1)  # Add a pseudo-sequence length of 1
        sentence_output, (hn, cn) = self.sent_bilstm(sentence_embedding)

        # Aggregate forward and backward hidden states
        sentence_output = sentence_output.squeeze(1)

        # Apply dropout
        x = self.dropout(sentence_output)

        # Compute outputs for each target
        outputs = {target: head(x) for target, head in self.output_heads.items()}
        return outputs

# Instantiate the model
input_dim = X.shape[1]  # Number of features per word embedding
hidden_dim = 64         # Number of features in LSTM hidden state
output_dims = {target: num_classes for target, num_classes in target_columns.items()}
model = HierarchicalBiLSTM(input_dim, hidden_dim, output_dims)

# Define loss functions for each output
criteria = {
    target: nn.CrossEntropyLoss()  # Adjust class weights if necessary
    for target in target_columns
}

# Define optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Move the model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Training function
def train_model(model, train_loader, criteria, optimizer, epochs=10):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            input_data = batch['input'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_data)

            loss = 0
            for idx, target in enumerate(target_columns.keys()):
                start = sum(list(target_columns.values())[:idx])
                end = start + target_columns[target]
                target_labels = torch.argmax(labels[:, start:end], dim=1)
                loss += criteria[target](outputs[target], target_labels)

            total_loss += loss.item()
            loss.backward()
            optimizer.step()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.4f}")

# Train the model
train_model(model, train_loader, criteria, optimizer, epochs=20)

# Evaluation function
def evaluate_model(model, val_loader):
    model.eval()
    predictions = {target: [] for target in target_columns}
    true_labels = {target: [] for target in target_columns}

    with torch.no_grad():
        for batch in val_loader:
            input_data = batch['input'].to(device)
            labels = batch['label'].cpu().numpy()

            # Get predictions from the model
            outputs = model(input_data)

            # For each target, append predictions and true labels
            for idx, target in enumerate(target_columns.keys()):
                preds = outputs[target].cpu().numpy()
                predictions[target].append(preds)

                # Extract correct true labels based on the target's range
                start = sum(list(target_columns.values())[:idx])
                end = start + target_columns[target]
                true_labels[target].append(labels[:, start:end])

    # Concatenate predictions and true labels for each target
    for target in target_columns:
        predictions[target] = np.vstack(predictions[target])
        true_labels[target] = np.vstack(true_labels[target])

    return predictions, true_labels

# Function to compute metrics
def compute_metrics(predictions, true_labels, encoders, target_columns):
    for target, num_classes in target_columns.items():
        # Get predicted and true labels
        y_pred = np.argmax(predictions[target], axis=1)
        y_true = np.argmax(true_labels[target], axis=1)

        # Get target names from encoder
        encoder = encoders[target]
        target_names = [str(cls) for cls in encoder.categories_[0]]

        print(f"Computing metrics for '{target}'...")

        # Display the classification report
        print(f"Classification Report for '{target}':")
        print(classification_report(y_true, y_pred, target_names=target_names))

        # Compute additional metrics: micro, macro, and weighted averages
        precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='micro')
        print(f"Micro-average Precision: {precision:.4f}")
        print(f"Micro-average Recall: {recall:.4f}")
        print(f"Micro-average F1-score: {f1:.4f}")

        precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro')
        print(f"Macro-average Precision: {precision:.4f}")
        print(f"Macro-average Recall: {recall:.4f}")
        print(f"Macro-average F1-score: {f1:.4f}")

        precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')
        print(f"Weighted-average Precision: {precision:.4f}")
        print(f"Weighted-average Recall: {recall:.4f}")
        print(f"Weighted-average F1-score: {f1:.4f}")

        # Compute overall accuracy
        accuracy = accuracy_score(y_true, y_pred)
        print(f"Overall Accuracy for '{target}': {accuracy:.4f}\n")

# After training, evaluate the model and display metrics
predictions, true_labels = evaluate_model(model, val_loader)
compute_metrics(predictions, true_labels, encoders, target_columns)


Epoch 1/20, Loss: 2.4935
Epoch 2/20, Loss: 2.3571
Epoch 3/20, Loss: 2.3346
Epoch 4/20, Loss: 2.3201
Epoch 5/20, Loss: 2.3059
Epoch 6/20, Loss: 2.2976
Epoch 7/20, Loss: 2.2891
Epoch 8/20, Loss: 2.2808
Epoch 9/20, Loss: 2.2724
Epoch 10/20, Loss: 2.2619
Epoch 11/20, Loss: 2.2511
Epoch 12/20, Loss: 2.2390
Epoch 13/20, Loss: 2.2254
Epoch 14/20, Loss: 2.2112
Epoch 15/20, Loss: 2.1960
Epoch 16/20, Loss: 2.1789
Epoch 17/20, Loss: 2.1624
Epoch 18/20, Loss: 2.1396
Epoch 19/20, Loss: 2.1239
Epoch 20/20, Loss: 2.1034
Computing metrics for 'provokingviolence'...
Classification Report for 'provokingviolence':
              precision    recall  f1-score   support

           0       0.48      0.29      0.37      1975
           1       0.17      0.01      0.02       966
           2       0.64      0.82      0.72      5855
           3       0.73      0.73      0.73      2191

    accuracy                           0.64     10987
   macro avg       0.51      0.46      0.46     10987
weighted avg     

In [1]:
#mtm lstm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support

# Load the dataset
data = pd.read_csv('embedded_sbert.csv')
data['embedding'] = data['embedding'].apply(lambda x: np.fromstring(x[1:-1], sep=','))

X = np.array(data['embedding'].tolist())

# Define target columns and their respective number of classes
target_columns = {
    'provokingviolence': 4,
    'individualharrassment': 4,
    'emotionaldistress': 3
}

# One-hot encode each target column
encoded_targets = {}
encoders = {}
for col, num_classes in target_columns.items():
    encoder = OneHotEncoder(sparse_output=False, categories='auto')
    encoded = encoder.fit_transform(data[col].values.reshape(-1, 1))
    encoded_targets[col] = encoded
    encoders[col] = encoder  # Save encoder for inverse transformations if needed

# Concatenate one-hot encoded targets into a single array
y = np.hstack([encoded_targets[col] for col in target_columns])

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Reshape X to add a time dimension for MTM LSTM
X_train = X_train[:, np.newaxis, :]
X_val = X_val[:, np.newaxis, :]

# Create a custom Dataset class
class MultiOutputDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input': torch.tensor(self.features[idx], dtype=torch.float32),
            'label': torch.tensor(self.labels[idx], dtype=torch.float32)
        }

# Create DataLoaders
train_dataset = MultiOutputDataset(X_train, y_train)
val_dataset = MultiOutputDataset(X_val, y_val)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Define the MTM LSTM model
class MTMLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dims, num_layers=1):
        super(MTMLSTM, self).__init__()

        # LSTM layer
        self.lstm = nn.LSTM(
            input_dim, hidden_dim, num_layers,
            batch_first=True, bidirectional=True
        )

        # Define output heads for each target
        self.output_heads = nn.ModuleDict({
            target: nn.Linear(hidden_dim * 2, output_dim)
            for target, output_dim in output_dims.items()
        })

        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        # LSTM output
        lstm_out, _ = self.lstm(x)

        # Apply dropout
        lstm_out = self.dropout(lstm_out)

        # Compute outputs for each target
        outputs = {target: head(lstm_out[:, -1, :]) for target, head in self.output_heads.items()}
        return outputs

# Instantiate the model
input_dim = X.shape[1]  # Number of features per word embedding
hidden_dim = 64         # Number of features in LSTM hidden state
output_dims = {target: num_classes for target, num_classes in target_columns.items()}
model = MTMLSTM(input_dim, hidden_dim, output_dims)

# Define loss functions for each output
criteria = {
    target: nn.CrossEntropyLoss()
    for target in target_columns
}

# Define optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Move the model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Training function
def train_model(model, train_loader, criteria, optimizer, epochs=10):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            input_data = batch['input'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_data)

            # Compute loss for each target
            loss = 0
            for idx, target in enumerate(target_columns.keys()):
                start = sum(list(target_columns.values())[:idx])
                end = start + target_columns[target]
                target_labels = torch.argmax(labels[:, start:end], dim=1)
                loss += criteria[target](outputs[target], target_labels)

            total_loss += loss.item()
            loss.backward()
            optimizer.step()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.4f}")

# Train the model
train_model(model, train_loader, criteria, optimizer, epochs=10)

# Evaluation function
def evaluate_model(model, val_loader):
    model.eval()
    predictions = {target: [] for target in target_columns}
    true_labels = {target: [] for target in target_columns}

    with torch.no_grad():
        for batch in val_loader:
            input_data = batch['input'].to(device)
            labels = batch['label'].cpu().numpy()

            outputs = model(input_data)
            for idx, target in enumerate(target_columns.keys()):
                preds = outputs[target].cpu().numpy()
                predictions[target].append(preds)

                start = sum(list(target_columns.values())[:idx])
                end = start + target_columns[target]
                true_labels[target].append(labels[:, start:end])

    for target in target_columns:
        predictions[target] = np.vstack(predictions[target])
        true_labels[target] = np.vstack(true_labels[target])

    return predictions, true_labels

# Evaluate the model
predictions, true_labels = evaluate_model(model, val_loader)

# Function to compute metrics
def compute_metrics(predictions, true_labels, encoders, target_columns):
    for target, num_classes in target_columns.items():
        y_pred = np.argmax(predictions[target], axis=1)
        y_true = np.argmax(true_labels[target], axis=1)

        encoder = encoders[target]
        target_names = [str(cls) for cls in encoder.categories_[0]]

        print(f"Classification Report for '{target}':")
        print(classification_report(y_true, y_pred, target_names=target_names))

        # Compute additional metrics: micro, macro, and weighted averages
        precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='micro')
        print(f"Micro-average Precision: {precision:.4f}")
        print(f"Micro-average Recall: {recall:.4f}")
        print(f"Micro-average F1-score: {f1:.4f}")

        precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro')
        print(f"Macro-average Precision: {precision:.4f}")
        print(f"Macro-average Recall: {recall:.4f}")
        print(f"Macro-average F1-score: {f1:.4f}")

        precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')
        print(f"Weighted-average Precision: {precision:.4f}")
        print(f"Weighted-average Recall: {recall:.4f}")
        print(f"Weighted-average F1-score: {f1:.4f}")

        # Compute overall accuracy
        accuracy = accuracy_score(y_true, y_pred)
        print(f"Overall Accuracy for '{target}': {accuracy:.4f}\n")

# Display the metrics
compute_metrics(predictions, true_labels, encoders, target_columns)

# Save the multi-class classification model
torch.save(model.state_dict(), 'multi_class_model.pth')

Epoch 1/10, Loss: 2.5030
Epoch 2/10, Loss: 2.3642
Epoch 3/10, Loss: 2.3459
Epoch 4/10, Loss: 2.3332
Epoch 5/10, Loss: 2.3204
Epoch 6/10, Loss: 2.3114
Epoch 7/10, Loss: 2.3039
Epoch 8/10, Loss: 2.2998
Epoch 9/10, Loss: 2.2925
Epoch 10/10, Loss: 2.2871
Classification Report for 'provokingviolence':
              precision    recall  f1-score   support

           0       0.55      0.20      0.30      1975
           1       0.00      0.00      0.00       966
           2       0.62      0.88      0.73      5855
           3       0.76      0.69      0.73      2191

    accuracy                           0.64     10987
   macro avg       0.48      0.44      0.44     10987
weighted avg       0.58      0.64      0.59     10987

Micro-average Precision: 0.6429
Micro-average Recall: 0.6429
Micro-average F1-score: 0.6429
Macro-average Precision: 0.4829
Macro-average Recall: 0.4435
Macro-average F1-score: 0.4379
Weighted-average Precision: 0.5822
Weighted-average Recall: 0.6429
Weighted-average

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/m

In [ ]:
#mlp
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import OneHotEncoder

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the dataset
data = pd.read_csv('embedded_sbert.csv')

# Convert the 'embedding' column to numpy arrays
data['embedding'] = data['embedding'].apply(lambda x: np.fromstring(x[1:-1], sep=','))
X = np.array(data['embedding'].tolist())

# Define target columns and their respective number of classes
target_columns = {
    'provokingviolence': 4,
    'individualharrassment': 4,
    'emotionaldistress': 3
}

# One-hot encode each target column
encoded_targets = {}
encoders = {}
for col, num_classes in target_columns.items():
    encoder = OneHotEncoder(sparse_output=False)
    encoded = encoder.fit_transform(data[col].values.reshape(-1, 1))
    encoded_targets[col] = encoded
    encoders[col] = encoder

# Concatenate one-hot encoded targets into a single array
y = np.hstack([encoded_targets[col] for col in target_columns])

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Custom Dataset class
class MultiOutputDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input': torch.tensor(self.features[idx], dtype=torch.float32),
            'label': torch.tensor(self.labels[idx], dtype=torch.float32)
        }

# Create DataLoaders
train_dataset = MultiOutputDataset(X_train, y_train)
val_dataset = MultiOutputDataset(X_val, y_val)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

# Define the Multi-Output MLP model
class MultiOutputMLPClassifier(nn.Module):
    def __init__(self, input_dim, output_dims):
        super(MultiOutputMLPClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 64)
        self.fc5 = nn.Linear(64, 32)

        # Separate output heads for each target
        self.output_heads = nn.ModuleDict({
            target: nn.Linear(32, output_dim)
            for target, output_dim in output_dims.items()
        })

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = torch.relu(self.fc4(x))
        x = torch.relu(self.fc5(x))

        outputs = {target: head(x) for target, head in self.output_heads.items()}
        return outputs

# Instantiate the model and move it to the specified device
input_dim = X.shape[1]
output_dims = {col: num_classes for col, num_classes in target_columns.items()}
model = MultiOutputMLPClassifier(input_dim, output_dims).to(device)

# Define weighted loss functions for each output
weights = {
    'provokingviolence': torch.tensor([0.5, 1.5, 0.7, 0.7], dtype=torch.float32).to(device),
    'individualharrassment': torch.tensor([1.0, 0.8, 0.7, 0.9], dtype=torch.float32).to(device),
    'emotionaldistress': torch.tensor([1.0, 0.8, 0.5], dtype=torch.float32).to(device)
}

criteria = {
    target: nn.BCEWithLogitsLoss(weight=weights[target])
    for target in target_columns
}

# Define optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training function
def train_model(model, train_loader, criteria, optimizer, epochs=30):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            input_data = batch['input'].to(device)
            labels = batch['label'].to(device)

            # Forward pass
            outputs = model(input_data)

            # Compute loss for each target
            loss = 0
            for idx, target in enumerate(target_columns.keys()):
                start = sum(list(target_columns.values())[:idx])
                end = start + target_columns[target]
                target_labels = labels[:, start:end]
                loss += criteria[target](outputs[target], target_labels)

            total_loss += loss.item()

            # Backward pass
            loss.backward()
            optimizer.step()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.4f}")

# Train the model
train_model(model, train_loader, criteria, optimizer, epochs=30)

# Evaluation function
def evaluate_model(model, val_loader):
    model.eval()
    predictions = {target: [] for target in target_columns}
    true_labels = {target: [] for target in target_columns}

    with torch.no_grad():
        for batch in val_loader:
            input_data = batch['input'].to(device)
            labels = batch['label'].cpu().numpy()

            outputs = model(input_data)
            for idx, target in enumerate(target_columns.keys()):
                preds = outputs[target].cpu().numpy()
                predictions[target].append(preds)

                start = sum(list(target_columns.values())[:idx])
                end = start + target_columns[target]
                true_labels[target].append(labels[:, start:end])

    for target in target_columns:
        predictions[target] = np.vstack(predictions[target])
        true_labels[target] = np.vstack(true_labels[target])

    return predictions, true_labels

# Evaluate the model
predictions, true_labels = evaluate_model(model, val_loader)

# Compute metrics for each target
def compute_metrics(predictions, true_labels, encoders, target_columns):
    for target, num_classes in target_columns.items():
        y_pred = (torch.sigmoid(torch.tensor(predictions[target])) > 0.5).int().numpy()
        y_true = true_labels[target]

        y_pred_labels = np.argmax(y_pred, axis=1)
        y_true_labels = np.argmax(y_true, axis=1)

        encoder = encoders[target]
        target_names = [str(cls) for cls in encoder.categories_[0]]

        print(f"Classification Report for '{target}':")
        print(classification_report(
            y_true_labels,
            y_pred_labels,
            target_names=target_names,
            zero_division=0,
            labels=np.unique(y_true_labels)
        ))

        # Calculate overall, weighted, macro, and micro accuracies
        accuracy = accuracy_score(y_true_labels, y_pred_labels)
        print(f"Overall Accuracy for '{target}': {accuracy:.4f}\n")

# Display the metrics
compute_metrics(predictions, true_labels, encoders, target_columns)


Epoch 1/30, Loss: 0.9108
Epoch 2/30, Loss: 0.8671
Epoch 3/30, Loss: 0.8564
Epoch 4/30, Loss: 0.8463
Epoch 5/30, Loss: 0.8378
Epoch 6/30, Loss: 0.8296
Epoch 7/30, Loss: 0.8181
Epoch 8/30, Loss: 0.8089
Epoch 9/30, Loss: 0.8022
Epoch 10/30, Loss: 0.7939
Epoch 11/30, Loss: 0.7875
Epoch 12/30, Loss: 0.7798
Epoch 13/30, Loss: 0.7730
Epoch 14/30, Loss: 0.7676
Epoch 15/30, Loss: 0.7635
Epoch 16/30, Loss: 0.7560
Epoch 17/30, Loss: 0.7522
Epoch 18/30, Loss: 0.7488
Epoch 19/30, Loss: 0.7474
Epoch 20/30, Loss: 0.7430
Epoch 21/30, Loss: 0.7424
Epoch 22/30, Loss: 0.7429
Epoch 23/30, Loss: 0.7334
Epoch 24/30, Loss: 0.7304
Epoch 25/30, Loss: 0.7268
Epoch 26/30, Loss: 0.7211
Epoch 27/30, Loss: 0.7199
Epoch 28/30, Loss: 0.7200
Epoch 29/30, Loss: 0.7159
Epoch 30/30, Loss: 0.7136
Classification Report for 'provokingviolence':
              precision    recall  f1-score   support

           0       0.40      0.43      0.41      1975
           1       0.20      0.04      0.06       966
           2       

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import xgboost as xgb
from sklearn.model_selection import GridSearchCV

# Load the dataset
data = pd.read_csv('embedded_sbert.csv')

# Convert the 'embedding' column to numpy arrays
data['embedding'] = data['embedding'].apply(lambda x: np.fromstring(x[1:-1], sep=','))
X = np.array(data['embedding'].tolist())

# Define the target columns and initialize label encoders for each
target_columns = ['provokingviolence', 'individualharrassment', 'emotionaldistress']
label_encoders = {col: LabelEncoder() for col in target_columns}

# Encode the labels for each target column
y_encoded = {}
for col in target_columns:
    y_encoded[col] = label_encoders[col].fit_transform(data[col])

# Split data into training and validation sets for each target column
train_test_splits = {}
for col in target_columns:
    X_train, X_val, y_train, y_val = train_test_split(X, y_encoded[col], test_size=0.2, random_state=42)
    train_test_splits[col] = (X_train, X_val, y_train, y_val)

# Function to train and evaluate XGBoost for each target
def train_evaluate_xgboost(target_column):
    X_train, X_val, y_train, y_val = train_test_splits[target_column]

    # Initialize XGBoost classifier with GPU support
    model = xgb.XGBClassifier(
        objective='multi:softmax',
        num_class=len(label_encoders[target_column].classes_),  # Number of classes for the target
        use_label_encoder=False,
        random_state=42,
        tree_method='gpu_hist',  # Enable GPU support
        gpu_id=0,  # Use the first GPU if available
        max_depth=6,
        learning_rate=0.1,
        n_estimators=100
    )

    # Train the model
    model.fit(X_train, y_train)

    # Predict on validation data
    y_pred = model.predict(X_val)

    # Convert predictions and true labels back to original labels
    y_pred_labels = label_encoders[target_column].inverse_transform(y_pred)
    y_val_labels = label_encoders[target_column].inverse_transform(y_val)

    # Print classification report and accuracy
    print(f"Classification Report for '{target_column}':")
    print(classification_report(y_val_labels, y_pred_labels))
    accuracy = accuracy_score(y_val_labels, y_pred_labels)
    print(f"Overall Accuracy for '{target_column}': {accuracy:.4f}\n")

    return model

# Train and evaluate XGBoost model for each target column
models = {}
for col in target_columns:
    print(f"Training and evaluating model for target: {col}")
    models[col] = train_evaluate_xgboost(col)

# Optionally: Hyperparameter Tuning using GridSearchCV for improving model accuracy
param_grid = {
    'max_depth': [6, 8, 10],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [50, 100, 200]
}

# Using GridSearchCV with cross-validation for each target
def tune_model_with_gridsearch(target_column):
    X_train, X_val, y_train, y_val = train_test_splits[target_column]

    # Initialize XGBoost classifier with GPU support
    model = xgb.XGBClassifier(
        objective='multi:softmax',
        num_class=len(label_encoders[target_column].classes_),
        use_label_encoder=False,
        tree_method='gpu_hist',  # Enable GPU support
        gpu_id=0,  # Use the first GPU
        random_state=42
    )

    # GridSearchCV for hyperparameter tuning
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring='accuracy',
        cv=3,  # 3-fold cross-validation
        verbose=1,
        n_jobs=-1
    )

    # Fit GridSearchCV
    grid_search.fit(X_train, y_train)

    # Best parameters and score
    print(f"Best parameters for {target_column}: {grid_search.best_params_}")
    print(f"Best cross-validation score for {target_column}: {grid_search.best_score_:.4f}")

    # Predict on validation data with the best model
    y_pred = grid_search.best_estimator_.predict(X_val)

    # Convert predictions and true labels back to original labels
    y_pred_labels = label_encoders[target_column].inverse_transform(y_pred)
    y_val_labels = label_encoders[target_column].inverse_transform(y_val)

    # Print classification report and accuracy
    print(f"Classification Report for '{target_column}' (with GridSearchCV):")
    print(classification_report(y_val_labels, y_pred_labels))
    accuracy = accuracy_score(y_val_labels, y_pred_labels)
    print(f"Overall Accuracy for '{target_column}': {accuracy:.4f}\n")

# Uncomment the line below to perform hyperparameter tuning
'''for col in target_columns:
    print(f"Tuning model for target: {col}")
    tune_model_with_gridsearch(col)'''


Training and evaluating model for target: provokingviolence


/usr/local/lib/python3.10/dist-packages/xgboost/core.py:158: UserWarning: [12:44:47] WARNING: /workspace/src/common/error_msg.cc:45: `gpu_id` is deprecated since2.0.0, use `device` instead. E.g. device=cpu/cuda/cuda:0
  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.10/dist-packages/xgboost/core.py:158: UserWarning: [12:44:47] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.10/dist-packages/xgboost/core.py:158: UserWarning: [12:44:47] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.10/dist-packages/xgboost/core.py:158: UserWarning: [12:44:52] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU t

Classification Report for 'provokingviolence':
              precision    recall  f1-score   support

           0       0.50      0.26      0.34      1975
           1       0.17      0.01      0.02       966
           2       0.63      0.85      0.72      5855
           3       0.75      0.72      0.74      2191

    accuracy                           0.64     10987
   macro avg       0.52      0.46      0.45     10987
weighted avg       0.59      0.64      0.60     10987

Overall Accuracy for 'provokingviolence': 0.6409

Training and evaluating model for target: individualharrassment


/usr/local/lib/python3.10/dist-packages/xgboost/core.py:158: UserWarning: [12:44:55] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.10/dist-packages/xgboost/core.py:158: UserWarning: [12:44:55] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.10/dist-packages/xgboost/core.py:158: UserWarning: [12:45:00] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)


Classification Report for 'individualharrassment':
              precision    recall  f1-score   support

           0       0.17      0.02      0.04        81
           1       0.53      0.33      0.41      2386
           2       0.54      0.76      0.63      5430
           3       0.56      0.33      0.42      3090

    accuracy                           0.54     10987
   macro avg       0.45      0.36      0.37     10987
weighted avg       0.54      0.54      0.52     10987

Overall Accuracy for 'individualharrassment': 0.5397

Training and evaluating model for target: emotionaldistress


/usr/local/lib/python3.10/dist-packages/xgboost/core.py:158: UserWarning: [12:45:04] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.10/dist-packages/xgboost/core.py:158: UserWarning: [12:45:04] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Classification Report for 'emotionaldistress':
              precision    recall  f1-score   support

           0       0.50      0.04      0.07       100
           1       0.60      0.42      0.50      3151
           2       0.79      0.90      0.84      7736

    accuracy                           0.75     10987
   macro avg       0.63      0.45      0.47     10987
weighted avg       0.73      0.75      0.73     10987

Overall Accuracy for 'emotionaldistress': 0.7518

Tuning model for target: provokingviolence
Fitting 3 folds for each of 27 candidates, totalling 81 fits


/usr/local/lib/python3.10/dist-packages/xgboost/core.py:158: UserWarning: [12:45:07] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.10/dist-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
#bilstm
import pandas as pd
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Bidirectional, Dropout, Reshape
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Load the dataset
data_file = 'embedded_sbert.csv'  # Adjust this to your dataset path
data = pd.read_csv(data_file)

# Assume your dataset has the embedding and the labels
embeddings = np.array(data['embedding'].apply(lambda x: np.fromstring(x[1:-1], sep=',').tolist()).tolist())  # Adjust for correct embedding extraction
labels = data[['provokingviolence', 'individualharrassment', 'emotionaldistress']]  # Adjust based on your actual column names

# Convert labels to categorical for each output
Y_provoking = pd.get_dummies(labels['provokingviolence']).values
Y_harassment = pd.get_dummies(labels['individualharrassment']).values
Y_distress = pd.get_dummies(labels['emotionaldistress']).values

# Split data into training and test sets
X_train, X_test, Y_train_provoking, Y_test_provoking = train_test_split(embeddings, Y_provoking, test_size=0.30, random_state=1)
_, _, Y_train_harassment, Y_test_harassment = train_test_split(embeddings, Y_harassment, test_size=0.30, random_state=1)
_, _, Y_train_distress, Y_test_distress = train_test_split(embeddings, Y_distress, test_size=0.30, random_state=1)

# Model architecture with multiple outputs
input_layer = Input(shape=(embeddings.shape[1],))  # The input size matches the embedding size
x = Reshape((1, embeddings.shape[1]))(input_layer)  # Reshape to 3D for LSTM (batch_size, 1, embedding_size)
x = Dropout(0.3)(x)  # Apply dropout to the embedding layer
x = Bidirectional(LSTM(300, dropout=0.3, recurrent_dropout=0.3))(x)  # BiLSTM layer with increased units

# Define separate output layers for each label
output_provoking = Dense(4, activation='softmax', name='provokingviolence')(x)
output_harassment = Dense(4, activation='softmax', name='individualharrassment')(x)
output_distress = Dense(3, activation='softmax', name='emotionaldistress')(x)

# Compile multi-output model with separate metrics for each output
model = Model(inputs=input_layer, outputs=[output_provoking, output_harassment, output_distress])
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics={
                  'provokingviolence': ['accuracy'],
                  'individualharrassment': ['accuracy'],
                  'emotionaldistress': ['accuracy']
              })


print(model.summary())

# Train the model
epochs = 20  # Increased epochs for better training
batch_size = 64
history = model.fit(
    X_train,
    [Y_train_provoking, Y_train_harassment, Y_train_distress],
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1,
    callbacks=[EarlyStopping(monitor='val_loss', min_delta=0.0001, patience=3)]
)

# Evaluate the model
test_results = model.evaluate(X_test, [Y_test_provoking, Y_test_harassment, Y_test_distress])
print(f"Evaluation Results: {test_results}")

# Predict and evaluate each output independently
preds_provoking, preds_harassment, preds_distress = model.predict(X_test)

# Convert predictions to binary for each label
preds_provoking_binary = (preds_provoking == preds_provoking.max(axis=1, keepdims=1)).astype(int)
preds_harassment_binary = (preds_harassment == preds_harassment.max(axis=1, keepdims=1)).astype(int)
preds_distress_binary = (preds_distress == preds_distress.max(axis=1, keepdims=1)).astype(int)

# Evaluate classification metrics
print("Classification Report for Provoking Violence:")
print(classification_report(Y_test_provoking.argmax(axis=1), preds_provoking_binary.argmax(axis=1)))

print("Classification Report for Individual Harassment:")
print(classification_report(Y_test_harassment.argmax(axis=1), preds_harassment_binary.argmax(axis=1)))

print("Classification Report for Emotional Distress:")
print(classification_report(Y_test_distress.argmax(axis=1), preds_distress_binary.argmax(axis=1)))


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5             │ (None, 384)            │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_1 (Reshape)       │ (None, 1, 384)         │              0 │ input_layer_5[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_2 (Dropout)       │ (None, 1, 384)         │              0 │ reshape_1[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ bidirectional_4           │ (None, 600)            │      1,644,000 │ dropout_2[0][0]        │
│ (Bidirectional)           │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ provokingviolence (Dense) │ (None, 4)              │          2,404 │ bidirectional_4[0][0]  │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ individualharrassment     │ (None, 4)              │          2,404 │ bidirectional_4[0][0]  │
│ (Dense)                   │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ emotionaldistress (Dense) │ (None, 3)              │          1,803 │ bidirectional_4[0][0]  │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 1,650,611 (6.30 MB)

 Trainable params: 1,650,611 (6.30 MB)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/20
541/541 ━━━━━━━━━━━━━━━━━━━━ 16s 15ms/step - emotionaldistress_accuracy: 0.7121 - individualharrassment_accuracy: 0.5069 - loss: 2.7612 - provokingviolence_accuracy: 0.5787 - val_emotionaldistress_accuracy: 0.7418 - val_individualharrassment_accuracy: 0.5447 - val_loss: 2.3773 - val_provokingviolence_accuracy: 0.6386
Epoch 2/20
541/541 ━━━━━━━━━━━━━━━━━━━━ 17s 14ms/step - emotionaldistress_accuracy: 0.7327 - individualharrassment_accuracy: 0.5296 - loss: 2.4535 - provokingviolence_accuracy: 0.6212 - val_emotionaldistress_accuracy: 0.7460 - val_individualharrassment_accuracy: 0.5434 - val_loss: 2.3623 - val_provokingviolence_accuracy: 0.6381
Epoch 3/20
541/541 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - emotionaldistress_accuracy: 0.7391 - individualharrassment_accuracy: 0.5326 - loss: 2.4240 - provokingviolence_accuracy: 0.6296 - val_emotionaldistress_accuracy: 0.7488 - val_individualharrassment_accuracy: 0.5442 - val_loss: 2.3482 - val_provokingviolence_accuracy: 0.6453
Epoch 

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/m

In [ ]:
import pandas as pd
import numpy as np
from keras.models import Model
from keras.layers import Input, Dense, Dropout
from sklearn.model_selection import train_test_split
from keras.callbacks import EarlyStopping
from keras import backend as K
from sklearn.metrics import classification_report

# Load your dataset
data = pd.read_csv('embedded_sbert.csv')  # Your dataset with precomputed embeddings

# Assuming 'embedded_text' contains lists of embeddings as strings
# Convert the string representations of lists to actual lists
X = np.array(data['embedding'].apply(lambda x: np.fromstring(x.strip("[]"), sep=',')).tolist())

# Check the shape of X after conversion
print(f"Shape of X after converting: {X.shape}")

# Prepare target variables as one-hot encoded arrays
Y_provoking = pd.get_dummies(labels['provokingviolence']).values
Y_harassment = pd.get_dummies(labels['individualharrassment']).values
Y_distress = pd.get_dummies(labels['emotionaldistress']).values

# Check shapes of Y as well
print(f"Shapes of Y: Provoking: {Y_provoking.shape}, Harassment: {Y_harassment.shape}, Distress: {Y_distress.shape}")

# Split the data
X_train, X_test, Y_train_provoking, Y_test_provoking = train_test_split(X, Y_provoking, test_size=0.3, random_state=42)
_, _, Y_train_harassment, Y_test_harassment = train_test_split(X, Y_harassment, test_size=0.3, random_state=42)
_, _, Y_train_distress, Y_test_distress = train_test_split(X, Y_distress, test_size=0.3, random_state=42)

# Build the multi-task LSTM model
input_layer = Input(shape=(X.shape[1],))  # Adjust input shape based on your embeddings
x = Dense(256, activation='relu')(input_layer)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)

# Output layers for each task
output_provoking = Dense(4, activation='softmax', name='provokingviolence')(x)
output_harassment = Dense(4, activation='softmax', name='individualharrassment')(x)
output_distress = Dense(3, activation='softmax', name='emotionaldistress')(x)

metrics = ['accuracy'] * 3
# Compile the model
model = Model(inputs=input_layer, outputs=[output_provoking, output_harassment, output_distress])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics = metrics)

# Train the model
epochs = 10
batch_size = 64
history = model.fit(
    X_train,
    [Y_train_provoking, Y_train_harassment, Y_train_distress],
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1,
    callbacks=[EarlyStopping(monitor='val_loss', min_delta=0.0001)]
)

# Evaluate the model
test_results = model.evaluate(X_test, [Y_test_provoking, Y_test_harassment, Y_test_distress])
print(f"Evaluation Results: {test_results}")

# Predict and evaluate
preds_provoking, preds_harassment, preds_distress = model.predict(X_test)

# Classification reports
print("Classification Report for Provoking Violence:")
print(classification_report(Y_test_provoking.argmax(axis=1), preds_provoking.argmax(axis=1)))

print("Classification Report for Individual Harassment:")
print(classification_report(Y_test_harassment.argmax(axis=1), preds_harassment.argmax(axis=1)))

print("Classification Report for Emotional Distress:")
print(classification_report(Y_test_distress.argmax(axis=1), preds_distress.argmax(axis=1)))

# Define the squared Euclidean distance function
def squared_euclidean_distance(y_true, y_pred):
    return K.sum(K.square(y_true - y_pred), axis=-1)

# Example usage of squared Euclidean distance
# This should be part of a custom metric if needed
# distance = squared_euclidean_distance(Y_test_provoking, preds_provoking)


Shape of X after converting: (54932, 384)
Shapes of Y: Provoking: (54932, 4), Harassment: (54932, 4), Distress: (54932, 3)
Epoch 1/10
541/541 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - emotionaldistress_accuracy: 0.7123 - individualharrassment_accuracy: 0.5121 - loss: 2.6762 - provokingviolence_accuracy: 0.5864 - val_emotionaldistress_accuracy: 0.7522 - val_individualharrassment_accuracy: 0.5567 - val_loss: 2.3287 - val_provokingviolence_accuracy: 0.6316
Epoch 2/10
541/541 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - emotionaldistress_accuracy: 0.7555 - individualharrassment_accuracy: 0.5406 - loss: 2.3452 - provokingviolence_accuracy: 0.6403 - val_emotionaldistress_accuracy: 0.7566 - val_individualharrassment_accuracy: 0.5549 - val_loss: 2.3024 - val_provokingviolence_accuracy: 0.6342
Epoch 3/10
541/541 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - emotionaldistress_accuracy: 0.7559 - individualharrassment_accuracy: 0.5388 - loss: 2.3381 - provokingviolence_accuracy: 0.6366 - val_emotionaldistress_accuracy: 0.7

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/m

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder

# Set CUDA_LAUNCH_BLOCKING=1 for debugging
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

# Check if CUDA (GPU) is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load your dataset
data = pd.read_csv('embedded_sbert.csv')  # Your dataset with precomputed embeddings

# Convert 'embedding' column (string lists) to actual embeddings
X = np.array(data['embedding'].apply(lambda x: np.fromstring(x.strip("[]"), sep=',')).tolist())

# Prepare target variables (integer labels, not one-hot encoded)
le_provoking = LabelEncoder()
le_harassment = LabelEncoder()
le_distress = LabelEncoder()

Y_provoking = le_provoking.fit_transform(data['provokingviolence'])
Y_harassment = le_harassment.fit_transform(data['individualharrassment'])
Y_distress = le_distress.fit_transform(data['emotionaldistress'])

# Check for any invalid labels (e.g., NaN or unexpected values)
print(f"Unique labels for provoking violence: {np.unique(Y_provoking)}")
print(f"Unique labels for individual harassment: {np.unique(Y_harassment)}")
print(f"Unique labels for emotional distress: {np.unique(Y_distress)}")

# Split the data into training and test sets
X_train, X_test, Y_train_provoking, Y_test_provoking = train_test_split(X, Y_provoking, test_size=0.3, random_state=42)
_, _, Y_train_harassment, Y_test_harassment = train_test_split(X, Y_harassment, test_size=0.3, random_state=42)
_, _, Y_train_distress, Y_test_distress = train_test_split(X, Y_distress, test_size=0.3, random_state=42)

# --- Step 1: Define the model ---
class HybridNN(nn.Module):
    def __init__(self, input_dim):
        super(HybridNN, self).__init__()
        self.fc1 = nn.Linear(input_dim, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 64)

        # Output layers for each task
        self.fc_out_provoking = nn.Linear(64, 4)  # Adjust for the number of classes in 'provokingviolence'
        self.fc_out_harassment = nn.Linear(64, 4)  # Adjust for the number of classes in 'individualharrassment'
        self.fc_out_distress = nn.Linear(64, 3)    # Adjust for the number of classes in 'emotionaldistress'

        self.dropout = nn.Dropout(0.5)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = torch.relu(self.fc2(x))
        x = self.dropout(x)
        x = torch.relu(self.fc3(x))
        x = self.dropout(x)
        x = torch.relu(self.fc4(x))
        x = self.dropout(x)

        # Output for each task
        output_provoking = self.fc_out_provoking(x)
        output_harassment = self.fc_out_harassment(x)
        output_distress = self.fc_out_distress(x)

        return self.softmax(output_provoking), self.softmax(output_harassment), self.softmax(output_distress)

# --- Step 2: Prepare data for PyTorch ---
# Convert the input data and target labels to tensors and move them to GPU
X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
Y_train_provoking_tensor = torch.tensor(Y_train_provoking, dtype=torch.long).to(device)
Y_train_harassment_tensor = torch.tensor(Y_train_harassment, dtype=torch.long).to(device)
Y_train_distress_tensor = torch.tensor(Y_train_distress, dtype=torch.long).to(device)
Y_test_provoking_tensor = torch.tensor(Y_test_provoking, dtype=torch.long).to(device)
Y_test_harassment_tensor = torch.tensor(Y_test_harassment, dtype=torch.long).to(device)
Y_test_distress_tensor = torch.tensor(Y_test_distress, dtype=torch.long).to(device)

# --- Step 3: Initialize the model ---
input_dim = X_train.shape[1]  # Number of features in the input
model = HybridNN(input_dim).to(device)  # Move the model to GPU if available

# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# --- Step 4: Train the model ---
epochs = 20
batch_size = 64
train_size = X_train_tensor.size(0)

# Training loop
for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for i in range(0, train_size, batch_size):
        # Get mini-batch
        X_batch = X_train_tensor[i:i+batch_size]
        Y_batch_provoking = Y_train_provoking_tensor[i:i+batch_size]
        Y_batch_harassment = Y_train_harassment_tensor[i:i+batch_size]
        Y_batch_distress = Y_train_distress_tensor[i:i+batch_size]

        # Zero gradients
        optimizer.zero_grad()

        # Forward pass
        outputs_provoking, outputs_harassment, outputs_distress = model(X_batch)

        # Compute loss for each output
        loss_provoking = criterion(outputs_provoking, Y_batch_provoking)
        loss_harassment = criterion(outputs_harassment, Y_batch_harassment)
        loss_distress = criterion(outputs_distress, Y_batch_distress)

        # Total loss
        loss = loss_provoking + loss_harassment + loss_distress
        epoch_loss += loss.item()

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss/train_size:.4f}")

# --- Step 5: Evaluate the model ---
model.eval()

with torch.no_grad():
    outputs_provoking, outputs_harassment, outputs_distress = model(X_test_tensor)

    # Convert predictions to class labels
    preds_provoking = torch.argmax(outputs_provoking, dim=1).cpu().numpy()
    preds_harassment = torch.argmax(outputs_harassment, dim=1).cpu().numpy()
    preds_distress = torch.argmax(outputs_distress, dim=1).cpu().numpy()

# Classification reports
print("Classification Report for Provoking Violence:")
print(classification_report(Y_test_provoking, preds_provoking))

print("Classification Report for Individual Harassment:")
print(classification_report(Y_test_harassment, preds_harassment))

print("Classification Report for Emotional Distress:")
print(classification_report(Y_test_distress, preds_distress))


Using device: cuda
Unique labels for provoking violence: [0 1 2 3]
Unique labels for individual harassment: [0 1 2 3]
Unique labels for emotional distress: [0 1 2]


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
